# Neural Network Pipeline with Keras/TensorFlow

This notebook demonstrates building neural network pipelines using Keras and TensorFlow, covering:
1. **CNN (Convolutional Neural Networks)** - For image/spatial data
2. **RNN (Recurrent Neural Networks)** - For sequential/text/time-series data
3. **GAN (Generative Adversarial Networks)** - For generating synthetic data

Each architecture includes:
- Data loading and preprocessing
- Model building with configurable architectures
- Hyperparameter tuning using Keras Tuner
- Training with callbacks (early stopping, learning rate scheduling)
- Evaluation and visualization

## 1. Imports and Setup

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import os

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard

# Keras Tuner for hyperparameter optimization
!pip install -q keras-tuner
import keras_tuner as kt

# Sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Utilities
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Configure GPU memory growth (prevents OOM errors)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Configured {len(gpus)} GPU(s) for memory growth")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")

---
# Part A: Convolutional Neural Network (CNN)
## For Image Classification Tasks
---

## A.1 Load Image Data

In [ ]:
# Configuration for image data - UPDATE THESE FOR YOUR DATA
CNN_CONFIG = {
    'train_dir': 'data/images/train',      # Directory with training images
    'test_dir': 'data/images/test',        # Directory with test images
    'img_height': 224,                      # Target image height
    'img_width': 224,                       # Target image width
    'batch_size': 32,                       # Batch size for training
    'num_classes': 10,                      # Number of classes (auto-detected if using directory structure)
    'color_mode': 'rgb'                     # 'rgb' or 'grayscale'
}

# Alternative: Load from CSV with image paths
# train_df = pd.read_csv('data/train.csv')  # Columns: 'image_path', 'label'
# test_df = pd.read_csv('data/test.csv')

In [ ]:
# Option 1: Load images from directory structure
# Expected structure: train_dir/class_name/image.jpg

def load_images_from_directory(config):
    """Load images using ImageDataGenerator from directory structure."""
    
    # Data augmentation for training
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest',
        validation_split=0.2
    )
    
    # Only rescaling for test data
    test_datagen = ImageDataGenerator(rescale=1./255)
    
    # Load training data
    train_generator = train_datagen.flow_from_directory(
        config['train_dir'],
        target_size=(config['img_height'], config['img_width']),
        batch_size=config['batch_size'],
        color_mode=config['color_mode'],
        class_mode='categorical',
        subset='training',
        seed=RANDOM_STATE
    )
    
    # Load validation data
    val_generator = train_datagen.flow_from_directory(
        config['train_dir'],
        target_size=(config['img_height'], config['img_width']),
        batch_size=config['batch_size'],
        color_mode=config['color_mode'],
        class_mode='categorical',
        subset='validation',
        seed=RANDOM_STATE
    )
    
    # Load test data
    test_generator = test_datagen.flow_from_directory(
        config['test_dir'],
        target_size=(config['img_height'], config['img_width']),
        batch_size=config['batch_size'],
        color_mode=config['color_mode'],
        class_mode='categorical',
        shuffle=False
    )
    
    return train_generator, val_generator, test_generator

# Uncomment to load from directories:
# train_gen, val_gen, test_gen = load_images_from_directory(CNN_CONFIG)

In [ ]:
# Option 2: Load images from numpy arrays (e.g., MNIST, CIFAR)
# This example uses CIFAR-10 as a demonstration

def load_sample_image_data():
    """Load sample image dataset for demonstration."""
    
    # Load CIFAR-10 dataset
    (X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
    
    # Normalize pixel values to [0, 1]
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0
    
    # Convert labels to categorical
    num_classes = 10
    y_train = keras.utils.to_categorical(y_train, num_classes)
    y_test = keras.utils.to_categorical(y_test, num_classes)
    
    # Split training into train and validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=RANDOM_STATE
    )
    
    class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
    
    return (X_train, y_train), (X_val, y_val), (X_test, y_test), class_names

# Load sample data
(X_train_cnn, y_train_cnn), (X_val_cnn, y_val_cnn), (X_test_cnn, y_test_cnn), class_names_cnn = load_sample_image_data()

print(f"Training data shape: {X_train_cnn.shape}")
print(f"Validation data shape: {X_val_cnn.shape}")
print(f"Test data shape: {X_test_cnn.shape}")
print(f"Number of classes: {len(class_names_cnn)}")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(3, 5, figsize=(12, 8))
for i, ax in enumerate(axes.flatten()):
    idx = np.random.randint(0, len(X_train_cnn))
    ax.imshow(X_train_cnn[idx])
    ax.set_title(class_names_cnn[np.argmax(y_train_cnn[idx])])
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()

## A.2 Build CNN Model

In [ ]:
def build_cnn_model(input_shape, num_classes, hp=None):
    """
    Build a CNN model with optional hyperparameter tuning.
    
    Args:
        input_shape: Shape of input images (height, width, channels)
        num_classes: Number of output classes
        hp: Keras Tuner hyperparameters object (optional)
    """
    
    # Hyperparameters (with defaults or tuned values)
    if hp:
        num_conv_blocks = hp.Int('num_conv_blocks', min_value=2, max_value=4, default=3)
        filters_base = hp.Choice('filters_base', values=[32, 64], default=32)
        kernel_size = hp.Choice('kernel_size', values=[3, 5], default=3)
        dense_units = hp.Int('dense_units', min_value=128, max_value=512, step=128, default=256)
        dropout_rate = hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1, default=0.3)
        learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log', default=1e-3)
    else:
        num_conv_blocks = 3
        filters_base = 32
        kernel_size = 3
        dense_units = 256
        dropout_rate = 0.3
        learning_rate = 1e-3
    
    model = models.Sequential(name='CNN_Model')
    
    # Input layer
    model.add(layers.InputLayer(input_shape=input_shape))
    
    # Convolutional blocks
    for i in range(num_conv_blocks):
        filters = filters_base * (2 ** i)
        
        model.add(layers.Conv2D(
            filters=filters,
            kernel_size=kernel_size,
            padding='same',
            activation='relu',
            kernel_regularizer=regularizers.l2(1e-4)
        ))
        model.add(layers.BatchNormalization())
        model.add(layers.Conv2D(
            filters=filters,
            kernel_size=kernel_size,
            padding='same',
            activation='relu',
            kernel_regularizer=regularizers.l2(1e-4)
        ))
        model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D(pool_size=(2, 2)))
        model.add(layers.Dropout(dropout_rate / 2))
    
    # Flatten and dense layers
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(dense_units, activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(dropout_rate))
    
    # Output layer
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    # Compile model
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Build and display model architecture
input_shape_cnn = X_train_cnn.shape[1:]
num_classes_cnn = y_train_cnn.shape[1]

cnn_model = build_cnn_model(input_shape_cnn, num_classes_cnn)
cnn_model.summary()

## A.3 CNN Hyperparameter Tuning

In [ ]:
def cnn_model_builder(hp):
    """Model builder function for Keras Tuner."""
    return build_cnn_model(input_shape_cnn, num_classes_cnn, hp)

# Create tuner
cnn_tuner = kt.Hyperband(
    cnn_model_builder,
    objective='val_accuracy',
    max_epochs=20,
    factor=3,
    directory='tuner_results',
    project_name='cnn_tuning',
    overwrite=True
)

print("CNN Hyperparameter Search Space:")
cnn_tuner.search_space_summary()

In [ ]:
# Define callbacks for tuning
cnn_tuner_callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
]

# Run hyperparameter search
print("Starting CNN hyperparameter search...")
cnn_tuner.search(
    X_train_cnn, y_train_cnn,
    validation_data=(X_val_cnn, y_val_cnn),
    epochs=20,
    batch_size=32,
    callbacks=cnn_tuner_callbacks,
    verbose=1
)

print("\nCNN hyperparameter search complete!")

In [ ]:
# Get best hyperparameters
cnn_best_hps = cnn_tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best CNN Hyperparameters:")
print(f"  - num_conv_blocks: {cnn_best_hps.get('num_conv_blocks')}")
print(f"  - filters_base: {cnn_best_hps.get('filters_base')}")
print(f"  - kernel_size: {cnn_best_hps.get('kernel_size')}")
print(f"  - dense_units: {cnn_best_hps.get('dense_units')}")
print(f"  - dropout_rate: {cnn_best_hps.get('dropout_rate')}")
print(f"  - learning_rate: {cnn_best_hps.get('learning_rate'):.6f}")

## A.4 Train Best CNN Model

In [ ]:
# Build model with best hyperparameters
best_cnn_model = cnn_tuner.hypermodel.build(cnn_best_hps)

# Define training callbacks
cnn_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_cnn_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

# Train the model
print("Training CNN with best hyperparameters...")
cnn_history = best_cnn_model.fit(
    X_train_cnn, y_train_cnn,
    validation_data=(X_val_cnn, y_val_cnn),
    epochs=50,
    batch_size=32,
    callbacks=cnn_callbacks,
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(cnn_history.history['accuracy'], label='Training Accuracy')
axes[0].plot(cnn_history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('CNN Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(cnn_history.history['loss'], label='Training Loss')
axes[1].plot(cnn_history.history['val_loss'], label='Validation Loss')
axes[1].set_title('CNN Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## A.5 CNN Evaluation

In [ ]:
# Evaluate on test set
cnn_test_loss, cnn_test_accuracy = best_cnn_model.evaluate(X_test_cnn, y_test_cnn, verbose=0)
print(f"CNN Test Loss: {cnn_test_loss:.4f}")
print(f"CNN Test Accuracy: {cnn_test_accuracy:.4f}")

# Generate predictions
cnn_predictions = best_cnn_model.predict(X_test_cnn)
cnn_pred_classes = np.argmax(cnn_predictions, axis=1)
cnn_true_classes = np.argmax(y_test_cnn, axis=1)

# Classification report
print("\nClassification Report:")
print(classification_report(cnn_true_classes, cnn_pred_classes, target_names=class_names_cnn))

In [ ]:
# Confusion matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(cnn_true_classes, cnn_pred_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names_cnn, yticklabels=class_names_cnn)
plt.title('CNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize some predictions
fig, axes = plt.subplots(3, 5, figsize=(15, 10))
for i, ax in enumerate(axes.flatten()):
    idx = np.random.randint(0, len(X_test_cnn))
    ax.imshow(X_test_cnn[idx])
    true_label = class_names_cnn[cnn_true_classes[idx]]
    pred_label = class_names_cnn[cnn_pred_classes[idx]]
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label}', color=color)
    ax.axis('off')
plt.suptitle('CNN Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.show()

---
# Part B: Recurrent Neural Network (RNN)
## For Sequential/Text Classification Tasks
---

## B.1 Load Sequential/Text Data

In [ ]:
# Configuration for text/sequence data - UPDATE THESE FOR YOUR DATA
RNN_CONFIG = {
    'train_file': 'data/train.csv',         # CSV with 'text' and 'label' columns
    'test_file': 'data/test.csv',
    'text_column': 'text',                   # Column containing text
    'label_column': 'label',                 # Column containing labels
    'max_vocab_size': 20000,                 # Maximum vocabulary size
    'max_sequence_length': 200,              # Maximum sequence length
    'embedding_dim': 128                     # Embedding dimension
}

In [ ]:
# Option 1: Load text data from CSV
def load_text_from_csv(config):
    """Load and preprocess text data from CSV files."""
    
    train_df = pd.read_csv(config['train_file'])
    test_df = pd.read_csv(config['test_file'])
    
    texts_train = train_df[config['text_column']].astype(str).values
    labels_train = train_df[config['label_column']].values
    texts_test = test_df[config['text_column']].astype(str).values
    
    # Encode labels
    label_encoder = LabelEncoder()
    labels_train = label_encoder.fit_transform(labels_train)
    
    return texts_train, labels_train, texts_test, label_encoder

# Uncomment to load from CSV:
# texts_train, labels_train, texts_test, label_encoder = load_text_from_csv(RNN_CONFIG)

In [ ]:
# Option 2: Load sample text dataset (IMDB reviews)
def load_sample_text_data(max_vocab_size=20000, max_length=200):
    """Load IMDB dataset for demonstration."""
    
    # Load IMDB dataset
    (X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(num_words=max_vocab_size)
    
    # Pad sequences to fixed length
    X_train = pad_sequences(X_train, maxlen=max_length, padding='post', truncating='post')
    X_test = pad_sequences(X_test, maxlen=max_length, padding='post', truncating='post')
    
    # Split training into train and validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=RANDOM_STATE
    )
    
    return (X_train, y_train), (X_val, y_val), (X_test, y_test), max_vocab_size

# Load sample data
(X_train_rnn, y_train_rnn), (X_val_rnn, y_val_rnn), (X_test_rnn, y_test_rnn), vocab_size_rnn = load_sample_text_data()

print(f"Training data shape: {X_train_rnn.shape}")
print(f"Validation data shape: {X_val_rnn.shape}")
print(f"Test data shape: {X_test_rnn.shape}")
print(f"Vocabulary size: {vocab_size_rnn}")
print(f"Sequence length: {X_train_rnn.shape[1]}")

## B.2 Build RNN Model

In [ ]:
def build_rnn_model(vocab_size, max_length, num_classes=1, hp=None):
    """
    Build an RNN model (LSTM/GRU) with optional hyperparameter tuning.
    
    Args:
        vocab_size: Size of vocabulary
        max_length: Maximum sequence length
        num_classes: Number of output classes (1 for binary)
        hp: Keras Tuner hyperparameters object (optional)
    """
    
    # Hyperparameters
    if hp:
        rnn_type = hp.Choice('rnn_type', values=['LSTM', 'GRU', 'Bidirectional_LSTM', 'Bidirectional_GRU'])
        embedding_dim = hp.Choice('embedding_dim', values=[64, 128, 256], default=128)
        rnn_units = hp.Int('rnn_units', min_value=64, max_value=256, step=64, default=128)
        num_rnn_layers = hp.Int('num_rnn_layers', min_value=1, max_value=3, default=2)
        dense_units = hp.Int('dense_units', min_value=32, max_value=128, step=32, default=64)
        dropout_rate = hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1, default=0.3)
        learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log', default=1e-3)
    else:
        rnn_type = 'Bidirectional_LSTM'
        embedding_dim = 128
        rnn_units = 128
        num_rnn_layers = 2
        dense_units = 64
        dropout_rate = 0.3
        learning_rate = 1e-3
    
    model = models.Sequential(name='RNN_Model')
    
    # Embedding layer
    model.add(layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_length
    ))
    model.add(layers.SpatialDropout1D(dropout_rate))
    
    # RNN layers
    for i in range(num_rnn_layers):
        return_sequences = (i < num_rnn_layers - 1)  # Return sequences for all but last RNN layer
        
        if rnn_type == 'LSTM':
            rnn_layer = layers.LSTM(rnn_units, return_sequences=return_sequences, dropout=dropout_rate)
        elif rnn_type == 'GRU':
            rnn_layer = layers.GRU(rnn_units, return_sequences=return_sequences, dropout=dropout_rate)
        elif rnn_type == 'Bidirectional_LSTM':
            rnn_layer = layers.Bidirectional(
                layers.LSTM(rnn_units, return_sequences=return_sequences, dropout=dropout_rate)
            )
        else:  # Bidirectional_GRU
            rnn_layer = layers.Bidirectional(
                layers.GRU(rnn_units, return_sequences=return_sequences, dropout=dropout_rate)
            )
        
        model.add(rnn_layer)
    
    # Global pooling if returning sequences from last layer
    # model.add(layers.GlobalMaxPooling1D())  # Uncomment if return_sequences=True for last layer
    
    # Dense layers
    model.add(layers.Dense(dense_units, activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(dropout_rate))
    
    # Output layer
    if num_classes == 1:
        model.add(layers.Dense(1, activation='sigmoid'))
        loss = 'binary_crossentropy'
    else:
        model.add(layers.Dense(num_classes, activation='softmax'))
        loss = 'sparse_categorical_crossentropy'
    
    # Compile model
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss=loss,
        metrics=['accuracy']
    )
    
    return model

# Build and display model architecture
max_length_rnn = X_train_rnn.shape[1]

rnn_model = build_rnn_model(vocab_size_rnn, max_length_rnn)
rnn_model.summary()

## B.3 RNN Hyperparameter Tuning

In [ ]:
def rnn_model_builder(hp):
    """Model builder function for Keras Tuner."""
    return build_rnn_model(vocab_size_rnn, max_length_rnn, num_classes=1, hp=hp)

# Create tuner
rnn_tuner = kt.Hyperband(
    rnn_model_builder,
    objective='val_accuracy',
    max_epochs=15,
    factor=3,
    directory='tuner_results',
    project_name='rnn_tuning',
    overwrite=True
)

print("RNN Hyperparameter Search Space:")
rnn_tuner.search_space_summary()

In [ ]:
# Define callbacks for tuning
rnn_tuner_callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

# Run hyperparameter search
print("Starting RNN hyperparameter search...")
rnn_tuner.search(
    X_train_rnn, y_train_rnn,
    validation_data=(X_val_rnn, y_val_rnn),
    epochs=15,
    batch_size=64,
    callbacks=rnn_tuner_callbacks,
    verbose=1
)

print("\nRNN hyperparameter search complete!")

In [ ]:
# Get best hyperparameters
rnn_best_hps = rnn_tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best RNN Hyperparameters:")
print(f"  - rnn_type: {rnn_best_hps.get('rnn_type')}")
print(f"  - embedding_dim: {rnn_best_hps.get('embedding_dim')}")
print(f"  - rnn_units: {rnn_best_hps.get('rnn_units')}")
print(f"  - num_rnn_layers: {rnn_best_hps.get('num_rnn_layers')}")
print(f"  - dense_units: {rnn_best_hps.get('dense_units')}")
print(f"  - dropout_rate: {rnn_best_hps.get('dropout_rate')}")
print(f"  - learning_rate: {rnn_best_hps.get('learning_rate'):.6f}")

## B.4 Train Best RNN Model

In [ ]:
# Build model with best hyperparameters
best_rnn_model = rnn_tuner.hypermodel.build(rnn_best_hps)

# Define training callbacks
rnn_callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_rnn_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

# Train the model
print("Training RNN with best hyperparameters...")
rnn_history = best_rnn_model.fit(
    X_train_rnn, y_train_rnn,
    validation_data=(X_val_rnn, y_val_rnn),
    epochs=30,
    batch_size=64,
    callbacks=rnn_callbacks,
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(rnn_history.history['accuracy'], label='Training Accuracy')
axes[0].plot(rnn_history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('RNN Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(rnn_history.history['loss'], label='Training Loss')
axes[1].plot(rnn_history.history['val_loss'], label='Validation Loss')
axes[1].set_title('RNN Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## B.5 RNN Evaluation

In [ ]:
# Evaluate on test set
rnn_test_loss, rnn_test_accuracy = best_rnn_model.evaluate(X_test_rnn, y_test_rnn, verbose=0)
print(f"RNN Test Loss: {rnn_test_loss:.4f}")
print(f"RNN Test Accuracy: {rnn_test_accuracy:.4f}")

# Generate predictions
rnn_predictions = (best_rnn_model.predict(X_test_rnn) > 0.5).astype(int).flatten()

# Classification report
print("\nClassification Report:")
print(classification_report(y_test_rnn, rnn_predictions, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test_rnn, rnn_predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.title('RNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

---
# Part C: Generative Adversarial Network (GAN)
## For Generating Synthetic Data
---

## C.1 Load Data for GAN

In [ ]:
# Configuration for GAN
GAN_CONFIG = {
    'img_height': 28,
    'img_width': 28,
    'channels': 1,
    'latent_dim': 100,           # Dimension of random noise input
    'batch_size': 128,
    'epochs': 50
}

img_shape = (GAN_CONFIG['img_height'], GAN_CONFIG['img_width'], GAN_CONFIG['channels'])

In [ ]:
# Load sample data for GAN (MNIST)
def load_gan_data():
    """Load and preprocess data for GAN training."""
    
    # Load MNIST dataset
    (X_train, _), (_, _) = keras.datasets.mnist.load_data()
    
    # Normalize to [-1, 1] (better for GAN training)
    X_train = X_train.astype('float32')
    X_train = (X_train - 127.5) / 127.5
    
    # Add channel dimension
    X_train = np.expand_dims(X_train, axis=-1)
    
    return X_train

X_train_gan = load_gan_data()
print(f"GAN Training data shape: {X_train_gan.shape}")
print(f"Value range: [{X_train_gan.min():.2f}, {X_train_gan.max():.2f}]")

In [ ]:
# Visualize sample training images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    idx = np.random.randint(0, len(X_train_gan))
    ax.imshow(X_train_gan[idx].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('Sample Training Images for GAN', fontsize=14)
plt.tight_layout()
plt.show()

## C.2 Build GAN Components

In [ ]:
def build_generator(latent_dim, img_shape, hp=None):
    """
    Build the Generator network.
    
    The generator takes random noise and generates fake images.
    """
    
    # Hyperparameters
    if hp:
        dense_units = hp.Choice('gen_dense_units', values=[128, 256, 512], default=256)
        dropout_rate = hp.Float('gen_dropout', min_value=0.2, max_value=0.4, default=0.3)
    else:
        dense_units = 256
        dropout_rate = 0.3
    
    model = models.Sequential(name='Generator')
    
    # Foundation for 7x7 image
    model.add(layers.Dense(dense_units * 7 * 7, input_dim=latent_dim))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Reshape((7, 7, dense_units)))
    
    # Upsample to 14x14
    model.add(layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(dropout_rate))
    
    # Upsample to 28x28
    model.add(layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))
    
    # Output layer
    model.add(layers.Conv2D(img_shape[-1], kernel_size=7, padding='same', activation='tanh'))
    
    return model

# Build and display generator
generator = build_generator(GAN_CONFIG['latent_dim'], img_shape)
generator.summary()

In [ ]:
def build_discriminator(img_shape, hp=None):
    """
    Build the Discriminator network.
    
    The discriminator classifies images as real or fake.
    """
    
    # Hyperparameters
    if hp:
        dropout_rate = hp.Float('disc_dropout', min_value=0.25, max_value=0.5, default=0.4)
    else:
        dropout_rate = 0.4
    
    model = models.Sequential(name='Discriminator')
    
    # Convolutional layers
    model.add(layers.Conv2D(64, kernel_size=3, strides=2, padding='same', input_shape=img_shape))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(dropout_rate))
    
    model.add(layers.Conv2D(128, kernel_size=3, strides=2, padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(dropout_rate))
    
    model.add(layers.Conv2D(256, kernel_size=3, strides=2, padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(dropout_rate))
    
    # Classification
    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid'))
    
    return model

# Build and display discriminator
discriminator = build_discriminator(img_shape)
discriminator.summary()

In [ ]:
class GAN(keras.Model):
    """
    Custom GAN model class for training.
    """
    
    def __init__(self, discriminator, generator, latent_dim):
        super().__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        self.d_loss_tracker = keras.metrics.Mean(name='d_loss')
        self.g_loss_tracker = keras.metrics.Mean(name='g_loss')
    
    @property
    def metrics(self):
        return [self.d_loss_tracker, self.g_loss_tracker]
    
    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super().compile()
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.loss_fn = loss_fn
    
    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        
        # Generate random noise
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
        
        # Generate fake images
        generated_images = self.generator(random_latent_vectors)
        
        # Combine with real images
        combined_images = tf.concat([generated_images, real_images], axis=0)
        
        # Labels: 0 for fake, 1 for real (with label smoothing)
        labels = tf.concat([
            tf.zeros((batch_size, 1)),
            tf.ones((batch_size, 1)) * 0.9  # Label smoothing
        ], axis=0)
        
        # Add random noise to labels (helps with training stability)
        labels += 0.05 * tf.random.uniform(tf.shape(labels))
        
        # Train the discriminator
        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)
        
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(grads, self.discriminator.trainable_weights))
        
        # Train the generator
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
        
        # Misleading labels (we want generator to fool discriminator)
        misleading_labels = tf.ones((batch_size, 1))
        
        with tf.GradientTape() as tape:
            predictions = self.discriminator(self.generator(random_latent_vectors))
            g_loss = self.loss_fn(misleading_labels, predictions)
        
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))
        
        # Update metrics
        self.d_loss_tracker.update_state(d_loss)
        self.g_loss_tracker.update_state(g_loss)
        
        return {
            'd_loss': self.d_loss_tracker.result(),
            'g_loss': self.g_loss_tracker.result()
        }

## C.3 GAN Hyperparameter Tuning

In [ ]:
# For GANs, hyperparameter tuning is typically done via grid search
# due to the complexity of the training dynamics

GAN_HYPERPARAMETERS = {
    'learning_rate': [0.0001, 0.0002, 0.0005],
    'beta_1': [0.5, 0.9],
    'latent_dim': [100, 128, 256]
}

print("GAN Hyperparameter Options:")
for param, values in GAN_HYPERPARAMETERS.items():
    print(f"  {param}: {values}")

# Selected hyperparameters (you can iterate through combinations)
SELECTED_LR = 0.0002
SELECTED_BETA1 = 0.5
SELECTED_LATENT_DIM = 100

## C.4 Train GAN

In [ ]:
# Rebuild models with selected hyperparameters
generator = build_generator(SELECTED_LATENT_DIM, img_shape)
discriminator = build_discriminator(img_shape)

# Create GAN
gan = GAN(
    discriminator=discriminator,
    generator=generator,
    latent_dim=SELECTED_LATENT_DIM
)

# Compile GAN
gan.compile(
    d_optimizer=optimizers.Adam(learning_rate=SELECTED_LR, beta_1=SELECTED_BETA1),
    g_optimizer=optimizers.Adam(learning_rate=SELECTED_LR, beta_1=SELECTED_BETA1),
    loss_fn=keras.losses.BinaryCrossentropy()
)

In [ ]:
class GANMonitor(keras.callbacks.Callback):
    """Callback to monitor GAN training by saving generated images."""
    
    def __init__(self, num_images=16, latent_dim=100):
        self.num_images = num_images
        self.latent_dim = latent_dim
        self.random_latent_vectors = tf.random.normal(shape=(num_images, latent_dim))
        self.generated_images_history = []
    
    def on_epoch_end(self, epoch, logs=None):
        # Generate images
        generated_images = self.model.generator(self.random_latent_vectors)
        generated_images = (generated_images * 127.5 + 127.5).numpy()  # Denormalize
        self.generated_images_history.append((epoch, generated_images))
        
        # Display every 10 epochs
        if (epoch + 1) % 10 == 0:
            fig, axes = plt.subplots(4, 4, figsize=(8, 8))
            for i, ax in enumerate(axes.flatten()):
                ax.imshow(generated_images[i].squeeze(), cmap='gray')
                ax.axis('off')
            plt.suptitle(f'Generated Images at Epoch {epoch + 1}')
            plt.tight_layout()
            plt.show()

# Create callback
gan_monitor = GANMonitor(num_images=16, latent_dim=SELECTED_LATENT_DIM)

In [ ]:
# Train GAN
print("Training GAN...")
gan_history = gan.fit(
    X_train_gan,
    epochs=GAN_CONFIG['epochs'],
    batch_size=GAN_CONFIG['batch_size'],
    callbacks=[gan_monitor],
    verbose=1
)

In [ ]:
# Plot training losses
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(gan_history.history['d_loss'], label='Discriminator Loss')
ax.plot(gan_history.history['g_loss'], label='Generator Loss')
ax.set_title('GAN Training Losses')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()

## C.5 GAN Evaluation and Generation

In [ ]:
# Generate new images
def generate_images(generator, num_images, latent_dim):
    """Generate new images using the trained generator."""
    random_latent_vectors = tf.random.normal(shape=(num_images, latent_dim))
    generated_images = generator(random_latent_vectors)
    generated_images = (generated_images * 127.5 + 127.5).numpy()  # Denormalize to [0, 255]
    return generated_images

# Generate 25 images
generated_images = generate_images(generator, 25, SELECTED_LATENT_DIM)

# Display generated images
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(generated_images[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('Final Generated Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize latent space interpolation
def interpolate_latent_space(generator, latent_dim, num_steps=10):
    """Generate images by interpolating between two random points in latent space."""
    
    # Two random points in latent space
    z1 = tf.random.normal(shape=(1, latent_dim))
    z2 = tf.random.normal(shape=(1, latent_dim))
    
    # Interpolate
    interpolated_images = []
    for alpha in np.linspace(0, 1, num_steps):
        z = z1 * (1 - alpha) + z2 * alpha
        img = generator(z)
        img = (img * 127.5 + 127.5).numpy()
        interpolated_images.append(img[0])
    
    return interpolated_images

# Generate interpolated images
interpolated = interpolate_latent_space(generator, SELECTED_LATENT_DIM, num_steps=10)

# Display
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(interpolated[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('Latent Space Interpolation', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Compare real vs generated images
fig, axes = plt.subplots(2, 8, figsize=(16, 4))

# Real images (top row)
for i in range(8):
    idx = np.random.randint(0, len(X_train_gan))
    real_img = (X_train_gan[idx] * 127.5 + 127.5)  # Denormalize
    axes[0, i].imshow(real_img.squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Real', fontsize=12)

# Generated images (bottom row)
gen_imgs = generate_images(generator, 8, SELECTED_LATENT_DIM)
for i in range(8):
    axes[1, i].imshow(gen_imgs[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Generated', fontsize=12)

plt.suptitle('Real vs Generated Images', fontsize=14)
plt.tight_layout()
plt.show()

---
# Part D: Model Comparison and Summary
---

In [ ]:
# Save all models
best_cnn_model.save('final_cnn_model.keras')
best_rnn_model.save('final_rnn_model.keras')
generator.save('final_gan_generator.keras')
discriminator.save('final_gan_discriminator.keras')

print("All models saved successfully!")

In [ ]:
print("="*70)
print("NEURAL NETWORK PIPELINE - SUMMARY")
print("="*70)

print("\n" + "-"*70)
print("CNN (Convolutional Neural Network) - Image Classification")
print("-"*70)
print(f"  Best Hyperparameters:")
print(f"    - Conv Blocks: {cnn_best_hps.get('num_conv_blocks')}")
print(f"    - Base Filters: {cnn_best_hps.get('filters_base')}")
print(f"    - Kernel Size: {cnn_best_hps.get('kernel_size')}")
print(f"    - Dense Units: {cnn_best_hps.get('dense_units')}")
print(f"    - Dropout Rate: {cnn_best_hps.get('dropout_rate')}")
print(f"    - Learning Rate: {cnn_best_hps.get('learning_rate'):.6f}")
print(f"  Test Accuracy: {cnn_test_accuracy:.4f}")

print("\n" + "-"*70)
print("RNN (Recurrent Neural Network) - Text Classification")
print("-"*70)
print(f"  Best Hyperparameters:")
print(f"    - RNN Type: {rnn_best_hps.get('rnn_type')}")
print(f"    - Embedding Dim: {rnn_best_hps.get('embedding_dim')}")
print(f"    - RNN Units: {rnn_best_hps.get('rnn_units')}")
print(f"    - RNN Layers: {rnn_best_hps.get('num_rnn_layers')}")
print(f"    - Dense Units: {rnn_best_hps.get('dense_units')}")
print(f"    - Dropout Rate: {rnn_best_hps.get('dropout_rate')}")
print(f"    - Learning Rate: {rnn_best_hps.get('learning_rate'):.6f}")
print(f"  Test Accuracy: {rnn_test_accuracy:.4f}")

print("\n" + "-"*70)
print("GAN (Generative Adversarial Network) - Image Generation")
print("-"*70)
print(f"  Hyperparameters:")
print(f"    - Latent Dimension: {SELECTED_LATENT_DIM}")
print(f"    - Learning Rate: {SELECTED_LR}")
print(f"    - Beta 1: {SELECTED_BETA1}")
print(f"  Final Generator Loss: {gan_history.history['g_loss'][-1]:.4f}")
print(f"  Final Discriminator Loss: {gan_history.history['d_loss'][-1]:.4f}")

print("\n" + "="*70)

## References

- [TensorFlow/Keras Documentation](https://www.tensorflow.org/api_docs/python/tf/keras)
- [Keras Tuner Documentation](https://keras.io/keras_tuner/)
- [CNN Guide - Stanford CS231n](https://cs231n.github.io/convolutional-networks/)
- [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)
- [GAN Tutorial - TensorFlow](https://www.tensorflow.org/tutorials/generative/dcgan)
- [Generative Adversarial Networks (Goodfellow et al., 2014)](https://arxiv.org/abs/1406.2661)